In [ ]:
#importing csv file
import pandas as pd
df = pd.read_csv('customer_shopping_behavior.csv')

In [ ]:
df.head() # top 5 rows

In [ ]:
#basic info of all columns
df.info()

In [ ]:
# Summary statistics using .describe()
df.describe(include="all") 
#df.describe() -> will give us normal statistics of numeric data

In [ ]:
# Checking if missing data or null values are present in the dataset
df.isnull().sum()

In [ ]:
# Imputing missing values in 'Review Rating' column with the MEDIAN rating of the product category
# We will do it according to the category, so that cloth's rating won't affect footware or etc.

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [37]:
# Renaming columns according to snake casing('lower_case') for better readability and documentation
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_")
df.rename(columns={'purchase_amount_(usd)' : 'purchase_amount'}) 

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Middle-Aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Young Adults,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Middle-Aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Young Adults,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Middle-Aged,365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly,Adults,7
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly,Middle-Aged,14
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly,Middle-Aged,90
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly,Adults,7


In [ ]:
df.columns

In [ ]:
# Create a new column age_group
# we will divide the age into n groups and give them labels
labels = ['Young Adults', 'Adults', 'Middle-Aged', 'Senior']

df['age_group'] = pd.qcut(df['age'], q=4, labels= labels)

# qcut -> cuts or divides data into q equal parts/percentage, and labels ka use karke lables dedenge
# qcut divides the data so that each group has roughly the same number of observations.
# 1st it will sort the data, then data ki total freq (like total rows) ko 'q' parts m divde kardega
# it's like ki 25% poplulation ek grp m 25% ek me, and so on

In [ ]:
df[['age', 'age_group']].head()

In [ ]:
# create new column purchase_frequency_days -> need to convert the text data into num for better calc.

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

df[['purchase_frequency_days', 'frequency_of_purchases']].head()

In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df.drop('promo_code_used', axis=1) #axis = 1 -> vertical

<h3>Connecting Python Script to MySQL

In [ ]:
from sqlalchemy import create_engine

In [ ]:
from sqlalchemy import text

# MySQL connection
username = "root"
password = "2268"
host = "localhost"
port = "3306"
database= "customer_shopping_behavior"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}")
with engine.connect() as conn:
    result = conn.execute(text("SHOW DATABASES;"))
    for row in result:
        print(row)

In [ ]:
# Connect to the server and build the missing room
with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS customer_shopping_behavior;"))
    print("Database created successfully!")

In [41]:
# MySQL connection
username = "root"
password = "2268"
host = "127.0.0.1"
port = "3306"
database= "customer_shopping_behavior"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

In [ ]:
# Write DataFrame to MySQL
table_name = "customer_behavior"

# Push dataframe to MySQL
df.to_sql(name=table_name, con=engine, if_exists="replace", index=False)

3900